# NNHIP
Looking to summarise activities and costs for 43 Places as a comparison 2024-25 YTD and 2025-26 YTD, as a breakdown by the following categories:
### A&E: 
  - A&E (type 1 and 2)
  - A&E (type other)
  - A&E (total)

### APC:  
  - APC (non-elective):
    - 0 LoS (SUS % Split)
    - 1+ LoS (SUS % Split)
  - APC (elective inpatients)
  - APC (elective total)
  - APC (Day case)

### OP:
  - OP (first)
  - OP (follow-up)
  - OP (procedures)

### Bed day estimates based on SuS average LOS:
  - elective bed days
  - non-elective bed days

### Average LoS:
  - elective bed days
  - non-elective bed days

### Total

## Set up

In [0]:
# should 'data check' calculations be performed?
global flag_data_check, flag_data_export

flag_data_check = False # should data checks be performed?
flag_data_export = True # should data be exported?

In [0]:
# imports
import pyspark.sql.functions as F
from pyspark.sql import Window as W
from pyspark.sql import DataFrame

In [0]:
# set path to practice geo data
lake = "udalstdatacuratedprod.dfs.core.windows.net"
container = "unrestricted"
folder = "/aggregated/UKHF/Appts_In_General_Practice/Prac_Level_Crosstab1/"
gp_path = "abfss://" + container + "@" + lake + folder

# https://udalstdatacuratedprod.dfs.core.windows.net/unrestricted/reference/UKHD/ODS_Replica/GP_Practices_Membership/ # this one isn't it
# https://udalstdatacuratedprod.dfs.core.windows.net/restricted/aggregated/DCF/CovidOCVC/OVC_GP_PCN_Lookup/
# reference/Internal/Reference/RightCare_practice_CCG_pcn_quarter_lookup/Published/1/
# reference/NHSD/GPSystems/Ref_Other_GP_System_Data/Published/2/
# https://udalstdatacuratedprod.dfs.core.windows.net/unrestricted/aggregated/UKHF/Appts_In_General_Practice/Prac_Level_Crosstab1/

# set path to OPA data
lake = "udalstdatacuratedprod.dfs.core.windows.net"
container = "restricted"
folder = "/patientlevel/MESH/OPA/OPA_Core_Daily/Published/1/"
opa_path = "abfss://" + container + "@" + lake + folder

# set path to ECDS data
lake = "udalstdatacuratedprod.dfs.core.windows.net"
container = "restricted"
folder = "/patientlevel/MESH/ECDS/EC_Core/Published/1/"
ecds_path = "abfss://" + container + "@" + lake + folder

# set path to APC data
lake = "udalstdatacuratedprod.dfs.core.windows.net"
container = "restricted"
folder = "/patientlevel/MESH/APC/APCS_Core_Daily/Published/1/"
apc_path = "abfss://" + container + "@" + lake + folder

# set path to lake mart
lake = "udalstdataanalysisprod.dfs.core.windows.net"
container = "analytics-projects"
folder = "/StrategyUnit/StrategyUnit/Evaluation/NNHIP/"
eval_path = "abfss://" + container + "@" + lake + folder

In [0]:
def summarise_counts(df: DataFrame, field: str, total_val: int) -> DataFrame:
    """
    Groups a dataset by a field, counts records, and computes percentages

    Parameters
    ----------
    df : DataFrame
        Input dataset
    field : str
        Column name to group by
    total_val : int
        Total number of records in df (used as denominator)
    """
    result = (
        df
        .groupBy(F.col(field))
        .agg(F.count("*").alias("count"))
        .withColumn("percentage", F.round((F.col("count") / F.lit(total_val)) * 100, 1))
        .orderBy(F.col("percentage").desc())
    )
    return result

# Geography
Need to link activity to the corresponding Neighbhourhood / Place.
Alex Lawless mentioned this is done via the patient's GP practice. As in, activity is grouped by registered practice and month, and then:
- GP practice -> PCN (via ODS lookups)
- PCN -> Place (via Excel lookup - to be done in R)

In [0]:
gp = (
    spark.read
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(gp_path)
    .select("GP_Code", "PCN_Code") # get the just the GP / PCN link
    .distinct() # remove duplicates
)

if flag_data_check:
    gp.limit(10).display()

In [0]:
if flag_data_export:
    gp.write.mode("overwrite").parquet(
        eval_path + "gp_pcn.parquet"
    )

# OPA
Outpatient attendances

In [0]:
opa = (
    spark.read
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(opa_path)
    .filter(F.col("Der_Activity_Month") >= 202404)
)

if flag_data_check:
    opa.limit(10).display()

## Due dilligence

In [0]:
if flag_data_check:
    total = opa.count()

In [0]:
if flag_data_check:
    result = summarise_counts(df = opa, field = "Der_Appointment_Type", total_val = total)
    display(result)

**Der_Appointment_Type**
- 66.9% `FUp`
- 32.6% `New`
- 0.4% `N/A`

In [0]:
if flag_data_check:
    result = summarise_counts(df = opa, field = "Der_Attendance_Type", total_val = total)
    display(result)

**Der_Attendance_Type**
- 77.4% `Attend`
- 9.2%  `Cancel (Hos)`
- 7.1%  `Cancel (Pat)`
- 5.5%  `DNA`
- 0.5%  `Unk`
- 0.3%  `Admin Event`

In [0]:
if flag_data_check:
    result = summarise_counts(df = opa, field = "Operation_Status", total_val = total)
    display(result)

**Operation_Status**
- 62.7% `Null`
- 25.7% `8` = "No operative procedures performed or intended"
- 9.1%  `1` = "1 or more operative procedures carried out"
- 2.9%  `9` = "Not known, i.e. finished episode but no data entered or the episode is unfinished and no data needs to be present (this would only be for a finished episode)"

In [0]:
if flag_data_check:
    result = summarise_counts(df = opa, field = "Der_Procedure_All", total_val = total)
    display(result)

**Der_Procedure_All**
This is not suitable - lists ? OPCS codes

In [0]:
if flag_data_check:
    result = summarise_counts(df = opa, field = "Der_Number_Procedure", total_val = total)
    display(result)

**Der_Number_Procedure**
- 72.1% `0`
- 14.4% `1`
- 6.7%  `2`
- 3.5%  `3`
- 1.9%  `4`
- 0.6%  `5`

... continues with incremental numbers of procedures, up to 110 procedures apparently conducted in one OPA case.
No `N/A` or `NULL` values

In [0]:
if flag_data_check:
    result = summarise_counts(df = opa, field = "Der_Staff_Type", total_val = total)
    display(result)

**Der_Staff_Type**
- 77.8% `Cons`
- 22.2% `Non-cons`

In [0]:
if flag_data_check:
    result = summarise_counts(df = opa, field = "Finished_Indicator", total_val = total)
    display(result)

**Finished_Indicator**
- 100%  `1`

## Calculations

**OP:**
  - OP (first)
  - OP (follow-up)
  - OP (procedures)

In [0]:
opa_first_fu = (
  opa
  .filter(F.col("Der_Appointment_Type").isin("New", "FUp")) # exclude N/A
  .filter(F.col("Der_Attendance_Type").isin("Attend")) # only count attendances
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code"), F.col("Der_Appointment_Type")) # figures by month, practice and attendance type
  .agg(F.countDistinct("OPA_Ident").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Attendances"))
)

if flag_data_check:
  opa_first_fu.limit(10).display()

In [0]:
if flag_data_export:
    opa_first_fu.write.mode("overwrite").parquet(
        eval_path + "opa_first_fu.parquet"
    )

In [0]:
opa_procedure = (
  opa
  .filter(F.col("Der_Attendance_Type").isin("Attend")) # only count attendances
  .filter(F.col("Der_Number_Procedure") > 0) # only count appointments where a procedure took place
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("OPA_Ident").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Attendances"))
)

if flag_data_check:
    opa_procedure.limit(10).display()

In [0]:
if flag_data_export:
    opa_procedure.write.mode("overwrite").parquet(
        eval_path + "opa_procedures.parquet"
    )

# A&E
  - A&E (type 1 and 2)
  - A&E (type other)
  - A&E (total)

## Load the data

In [0]:
ecds = (
    spark.read
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(ecds_path)
    .filter(F.col("Der_Activity_Month") >= 202404)
)

if flag_data_check:
    ecds.limit(10).display()

## Due dilligence

In [0]:
if flag_data_check:
    total = ecds.count()

In [0]:
if flag_data_check:
    result = summarise_counts(df = ecds, field = "EC_Department_Type", total_val = total)
    display(result)

**EC_Department_Type**
- 62.1% `01` = "Major Emergency Care Department"
- 30.4% `03` = "Urgent Treatment Centre"
- 4.6%  `05` = "Same Day Emergency Care Attendance"
- 1.9%  `02` = "Mono-specialty Emergency Care Department"
- 1%    `04` = "Other A&E Department Type"
- 0%    `99` = "?? not in the ToS"
- 0%    `06` = "Urgent and Emergency Care Extended Care Episode"
- 0%    `07` = "Hot clinic attendance: only valid for piloting purposes"

In [0]:
if flag_data_check:
    result = summarise_counts(df = ecds, field = "EC_Discharge_Status_Approved", total_val = total)
    display(result)

**EC_Discharge_Status_Approved**
- 91.4% `True`
- 8.3%  `null`
- 0.3%  `False`

In [0]:
if flag_data_check:
    result = summarise_counts(df = ecds, field = "EC_Discharge_Status_SNOMED_CT", total_val = total)
    display(result)

NB, there are 19 SNOMED CT codes in use

In [0]:
if flag_data_check:
    result = summarise_counts(df = ecds, field = "EC_AttendanceCategory", total_val = total)
    display(result)

**EC_AttendanceCategory**
- 91.3% `1` = "Unplanned first ED attendance"
- 4.8%  `null`
- 1.9%  `2` = "Unplanned follow-up ED attendance"
- 1.4%  `4` = "Planned follow-up ED attendance"
- 0.6%  `3` = "Unplanned follow-up ED attancdance"
- 0%    `X` = "Patient dead on arrival"

## Calculations
  - A&E (type 1 and 2)
  - A&E (type other)
  - A&E (total)

NB, these calculations were based on an NHSE guidance PDF called "ECDS Supplementary Analysis Technical Specification v3.3" (Nov 2025)

In [0]:
# NB, this is based on metric 2.1 - A&E attendances
ecds_ae_t1_t2 = (
  ecds
  .filter(F.col("EC_Department_Type").isin("01", "02")) # include type 1 and 2 attendances
  .filter(
    ~F.col("EC_Discharge_Status_SNOMED_CT").isin("63238001") | # exclude dead on arrival
    ~F.col("EC_AttendanceCategory").isin("X") # exclude dead on arrival
  )
  .filter(~F.col("EC_AttendanceCategory").isin("4")) # exclude planned follow-ups
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("EC_Ident").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Attendances"))
)

if flag_data_check:
    ecds_ae_t1_t2.limit(10).display()

In [0]:
if flag_data_export:
    ecds_ae_t1_t2.write.mode("overwrite").parquet(
        eval_path + "ecds_ae_t1_t2.parquet"
    )

In [0]:
# NB, this is based on metric 2.1 - A&E attendances
# i.e. NOT type 1 or 2
ecds_ae_tother = (
  ecds
  .filter(~F.col("EC_Department_Type").isin("01", "02")) # exclude type 1 and 2 attendances
  .filter(
    ~F.col("EC_Discharge_Status_SNOMED_CT").isin("63238001") | # exclude dead on arrival
    ~F.col("EC_AttendanceCategory").isin("X") # exclude dead on arrival
  )
  .filter(~F.col("EC_AttendanceCategory").isin("4")) # exclude planned follow-ups
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("EC_Ident").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Attendances"))
)

if flag_data_check:
    ecds_ae_tother.limit(10).display()

In [0]:
if flag_data_export:
    ecds_ae_tother.write.mode("overwrite").parquet(
        eval_path + "ecds_ae_tother.parquet"
    )

In [0]:
# NB, this is based on metric 2.1 - A&E attendances
# i.e. NOT type 1 or 2
ecds_ae_all = (
  ecds
  .filter(
    ~F.col("EC_Discharge_Status_SNOMED_CT").isin("63238001") | # exclude dead on arrival
    ~F.col("EC_AttendanceCategory").isin("X") # exclude dead on arrival
  )
  .filter(~F.col("EC_AttendanceCategory").isin("4")) # exclude planned follow-ups
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("EC_Ident").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Attendances"))
)

if flag_data_check:
    ecds_ae_all.limit(10).display()

In [0]:
if flag_data_export:
    ecds_ae_all.write.mode("overwrite").parquet(
        eval_path + "ecds_ae_all.parquet"
    )

# APC
  - APC (non-elective):
    - 0 LoS (SUS % Split)
    - 1+ LoS (SUS % Split)
  - APC (elective inpatients)
  - APC (elective total)
  - APC (Day case)

### Bed day estimates based on SuS average LOS:
  - elective bed days
  - non-elective bed days

### Average LoS:
  - elective bed days
  - non-elective bed days

In [0]:
apc = (
    spark.read
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(apc_path)
    .filter(F.col("Der_Activity_Month") >= 202404)
)

if flag_data_check:
    apc.limit(10).display()

## Due dilligence

In [0]:
if flag_data_check:
    total = apc.count()

In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Patient_Classification", total_val = total)
    display(result)

**Patient_Classification**
- 47.7%   `1` = Ordinary admission
- 43.1%   `2` = Day case admission
- 8.3%    `3` = Regular day attender
- 0.8%    `5` = Mothers and babies using only delivery facilities
- 0%    `8` = Not applicable (other maternity event)
- 0%    `null`
- 0%    `4` = Regular night attender

In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Admission_Method", total_val = total)
    display(result)

**Admission_Method**
- 27.2% `11` = Waiting list
- 24.3% `21` = Emergency Care Department
- 18.8% `13` = Planned
- 11.8% `12` = Booked
- 5.7% `31` = Admitted ante-partum
- 3.3% `2D` = Other emergency admission
- 3.2% `22` = General Practitioner
- 2.7% `82` = The birth of a baby
- 0.7% `24` = Consultant clinic
- 0.6% `81` = Non-emergency transfer
- 0.4% `28` = Other means
- 0.4% `32` = Admitted post-partum
- 0.3% `2A` = Emergency Care Department (another provider)
... total 21 rows of codes, includes `null`

In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Commissioner_Reference_Number", total_val = total)
    display(result)

10,000+ rows, lots of random codes, '59999' does not appear

In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Commissioner_Code", total_val = total)
    display(result)

1,106 rows, seem to be related to ODS codes. '59999' does not appear

In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Discharge_Method", total_val = total)
    display(result)

**Discharge_Method**
- 97.9% `1` Discharged on clinical advice or with clinical consent
- 1% `4` Patient died
- 0.4% `6` Patient discharged themself
- 0.3% `2` Patient discharged themself / discharged by relative or advocate
- 0.2% `9` Not known
- 0% `7` Discharged by relative or advocate
- 0% `3` Discharged by mental health review tribunal, home secretary or court
- 0% `8` Not applicable - Hospital Provider Spell not finished at episode (i.e. not discharged) or current episode unfinished
- 0% `null`
- 0% `5` Stillbirth

In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Der_Management_Type", total_val = total)
    display(result)

**Der_Management_Type**
- 43% `DC` = Day Case
- 32.7% `EM` = Emergency
- 9.4% `NE` = Non elective
- 8.2% `RDA` = Regular Day Attenders
- 6.6% `EL` = Elective
- 0.1% `UNK` = Unknown
- 0% `RNA` = Regular Night Attenders


In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Der_Spell_ID_Type", total_val = total)
    display(result)

All values `null`

In [0]:
if flag_data_check:
    result = summarise_counts(df = apc, field = "Der_Spell_Los", total_val = total)
    display(result)

**Der_Spell_LoS**
- 

In [0]:
# Which ID is the correct one to use for counting APC activity?
# Is it `APCS_Ident` - which appears to be the main key for the APCS data (NB, the 'S' signifies they've been worked into spells)
# Or is it `Der_Spell_ID`? This is a derived field and expicitly describes being a spell ID.

if flag_data_check:
    n_total = apc.count()
    n_apcs_ident = apc.select("APCS_Ident").distinct().count()
    n_der_spell_id = apc.select("Der_Spell_ID").distinct().count()
    n_pair = apc.select("APCS_Ident", "Der_Spell_ID").distinct().count()
    print(f"Total rows: {n_total}")
    print(f"Distinct APCS_Ident: {n_apcs_ident}")
    print(f"Distinct Der_Spell_ID: {n_der_spell_id}")
    print(f"Distinct (APCS_Ident, Der_Spell_ID) pairs: {n_pair}")
    if n_apcs_ident == n_der_spell_id == n_pair:
        print("APCS_Ident and Der_Spell_ID are equivalent and both uniquely identify spells.")
    else:
        print("APCS_Ident and Der_Spell_ID are not equivalent or do not both uniquely identify spells.")

## Calculations - Spells

These definitions are based on a file called 'Dashboard Data Descriptions.xlsx' from digital.nhs.uk 'Hospital Episode Statistics Data Dictionary'

In [0]:
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_elective_total = (
  apc
  .filter(F.col("Patient_Classification").isin("1", "2")) # Ordinary and Day Case
  .filter(F.col("Admission_Method").startswith("1")) # Elective admission methods (waiting list, booked, planned)
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("Der_Spell_ID").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Spells"))
)

if flag_data_check:
    apc_elective_total.limit(10).display()

In [0]:
if flag_data_export:
    apc_elective_total.write.mode("overwrite").parquet(
        eval_path + "apc_elective_total.parquet"
    )

In [0]:
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_elective_daycase = (
  apc
  .filter(F.col("Patient_Classification").isin("2")) # Day case
  .filter(F.col("Admission_Method").startswith("1")) # Elective admission methods (waiting list, booked, planned)
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("Der_Spell_ID").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Spells"))
)

if flag_data_check:
    apc_elective_daycase.limit(10).display()

In [0]:
if flag_data_export:
    apc_elective_daycase.write.mode("overwrite").parquet(
        eval_path + "apc_elective_daycase.parquet"
    )

In [0]:
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_elective_inpatient = (
  apc
  .filter(F.col("Patient_Classification").isin("1")) # Ordinary
  .filter(F.col("Admission_Method").startswith("1")) # Elective admission methods (waiting list, booked, planned)
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("Der_Spell_ID").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Spells"))
)

if flag_data_check:
    apc_elective_inpatient.limit(10).display()

In [0]:
if flag_data_export:
    apc_elective_inpatient.write.mode("overwrite").parquet(
        eval_path + "apc_elective_inpatient.parquet"
    )

In [0]:
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_nonelective = (
  apc
  .filter(F.col("Patient_Classification").isin("1")) # Ordinary
  .filter(
      (F.col("Admission_Method").startswith("2")) # Emergency admissions
      | (F.col("Admission_Method").startswith("3")) # Maternity
      | (F.col("Admission_Method").startswith("8")) # Other admission method (e.g. birth of baby, transfer from another hospital)
    )
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("Der_Spell_ID").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Spells"))
)

if flag_data_check:
    apc_nonelective.limit(10).display()

In [0]:
if flag_data_export:
    apc_nonelective.write.mode("overwrite").parquet(
        eval_path + "apc_nonelective.parquet"
    )

In [0]:
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_nonelective_lossplit = (
  apc
  .filter(F.col("Patient_Classification").isin("1")) # Ordinary
  .filter(
      (F.col("Admission_Method").startswith("2")) # Emergency admissions
      | (F.col("Admission_Method").startswith("3")) # Maternity
      | (F.col("Admission_Method").startswith("8")) # Other admission method (e.g. birth of baby, transfer from another hospital)
    )
  .withColumn(
      "LoS_category",
      F.when(F.col("Der_Spell_LoS") == 0, "0 LoS")
       .when(F.col("Der_Spell_LoS") > 0, "1+ LoS")
       .otherwise("Unknown")
  )
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code"), F.col("LoS_category")) # figures by month, practice, LoS category
  .agg(F.countDistinct("Der_Spell_ID").alias("n"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Spells"))
)

if flag_data_check:
    apc_nonelective_lossplit.limit(10).display()

In [0]:
if flag_data_export:
    apc_nonelective_lossplit.write.mode("overwrite").parquet(
        eval_path + "apc_nonelective_lossplit.parquet"
    )

## Calculations - average LoS
This is simply the average LoS for spells discharged in each month, split by practice and admission type (elective v non-elective)

Because the average can't be worked out at this level (i.e. at GP practice), the numerator and denominator will be stored and the average worked out in R once the data has been aggregated to Neighbourhood level.

Unresolved questions:
* Do I exclude zero LoS spells from this calculation?
* Is it based on mean or median?

In [0]:
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_average_los = (
    apc
    .filter(F.col("Patient_Classification").isin("1", "2")) # Ordinary and day case
    .withColumn(
        "calc_Admission_Type",
        F.when(F.col("Admission_Method").startswith("1"), "Elective") # Elective admissions
        .when(
            F.col("Admission_Method").startswith("2") # Emergency admissions
            | F.col("Admission_Method").startswith("3") # Maternity
            | F.col("Admission_Method").startswith("8"), # Other admission method (e.g. birth of baby, transfer from another hospital)
            "Non-Elective"
        )
        .otherwise("Unknown")
    )
    .filter(F.col("calc_Admission_Type").isin("Elective", "Non-Elective")) # focus on the types of interest
    .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code"), F.col("calc_Admission_Type")) # figures by month, practice, admission type
    .agg(F.countDistinct("Der_Spell_ID").alias("n"), F.sum("Der_Spell_LoS").alias("LoS"))
    .orderBy(F.col("Der_Activity_Month").asc())
    .withColumn("n_unit", F.lit("Spells"))
    .withColumn("los_unit", F.lit("Days"))
)

if flag_data_check:
    apc_average_los.limit(10).display()

In [0]:
if flag_data_export:
    apc_average_los.write.mode("overwrite").parquet(
        eval_path + "apc_average_los.parquet"
    )

This table is there reference value - average LoS per month and admission type. This will be used in R to work out the bed day estimates per neighbourhood, i.e.:

`neighbhourhood avg_los * mean_los`

In [0]:
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_ref_average_los = (
    apc
    .filter(F.col("Patient_Classification").isin("1", "2")) # Ordinary and day case
    .withColumn(
        "calc_Admission_Type",
        F.when(F.col("Admission_Method").startswith("1"), "Elective") # Elective admissions
        .when(
            F.col("Admission_Method").startswith("2") # Emergency admissions
            | F.col("Admission_Method").startswith("3") # Maternity
            | F.col("Admission_Method").startswith("8"), # Other admission method (e.g. birth of baby, transfer from another hospital)
            "Non-Elective"
        )
        .otherwise("Unknown")
    )
    .filter(F.col("calc_Admission_Type").isin("Elective", "Non-Elective")) # focus on the types of interest
    .groupBy(F.col("Der_Activity_Month"), F.col("calc_Admission_Type")) # figures by month, admission type
    .agg(F.countDistinct("Der_Spell_ID").alias("n"), F.sum("Der_Spell_LoS").alias("LoS"))
    .orderBy(F.col("Der_Activity_Month").asc())
    .withColumn("n_unit", F.lit("Spells"))
    .withColumn("los_unit", F.lit("Days"))
    .withColumn("mean_los", F.col("LoS") / F.col("n"))
)

if flag_data_check:
    apc_ref_average_los.limit(10).display()

In [0]:
if flag_data_export:
    apc_ref_average_los.write.mode("overwrite").parquet(
        eval_path + "apc_ref_average_los.parquet"
    )

## Calculations - Bed days
This is working on the following approach:
1. Work out the average (mean) length of stay by month and admission type (elective v non-elective)
2. Count spells per month, practice and admission type (elective v non-elective) (similar to above approach, or just load that data in)
3. Join average LOS to spells count on both month and admission type
4. Multiply average LoS by spell count to estimate bed days


In [0]:
%skip
# NB, this is based on 'Dashboard Data Descriptions.xlsx': Hospital Episode Statistics Monthly Activity Report
apc_beddays_elective_total = (
  apc
  .filter(F.col("Patient_Classification").isin("1", "2")) # Ordinary and Day Case
  # .filter(F.col("Admission_Method").startswith("1")) # Elective admission methods (waiting list, booked, planned)
  .withColumn("C")
  .groupBy(F.col("Der_Activity_Month"), F.col("GP_Practice_Code")) # figures by month, practice
  .agg(F.countDistinct("Der_Spell_ID").alias("n"), F.sum("Der_Spell_LoS").alias("bd"))
  .orderBy(F.col("Der_Activity_Month").asc())
  .withColumn("unit", F.lit("Bed Days"))
)

if flag_data_check:
    apc_beddays_elective_total.limit(10).display()